# Oldingi Mendeley datasetida tashqi baholash

Fayl: `06_evaluate_mendeley_external_dataset.ipynb`. Ishga tushirish va papkalar izohi: [README.md](README.md).

Bu tarixiy notebook Mendeley uchun yozilgan. `karakalpak_combined/train.csv` va audio manzilini sozlash kerak. Hozirgi Benchmark dataset uchun `08_finetune_and_evaluate_benchmark.ipynb`dan foydalaning.


In [2]:
import pandas as pd
from pathlib import Path

DATASET_DIR = Path(
    "/home/intern/projects/karakalpak_stt/"
    "external_validation/karakalpak_combined"
)

df = pd.read_csv(DATASET_DIR / "train.csv")

print("Total samples:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

Total samples: 3708
Columns: ['audio_path', 'text', 'duration']


,audio_path,text,duration
0,train/audio/hf_audio_000001.wav,kún uyasına sıymay dolanıp qıp qızıl shoqtay j...,6.952250
1,train/audio/hf_audio_000002.wav,kese etek ten qońıratqa qaray sozılǵan úlken g...,5.058687
2,train/audio/hf_audio_000003.wav,jeter jerde el joq,2.006562
3,train/audio/hf_audio_000004.wav,joldıń eki boyı qula dúz keń dala qula dúz ada...,9.693500
4,train/audio/hf_audio_000005.wav,hár jerde bir túp kógergen shópler ǵana kórinedi,3.617437


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATASET_DIR = Path(
    "/home/intern/projects/karakalpak_stt/"
    "external_validation/karakalpak_combined"
)

df = pd.read_csv(DATASET_DIR / "train.csv")

print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)

print("Total samples:", len(df))
print("Columns:", df.columns.tolist())

print("\nMISSING VALUES:")
print(df.isna().sum())

print("\nDUPLICATES:")
print("Duplicate audio paths:", df["audio_path"].duplicated().sum())
print("Duplicate texts:", df["text"].duplicated().sum())

print("\nDURATION STATISTICS:")
print(df["duration"].describe())

DATASET INFORMATION
Total samples: 3708
Columns: ['audio_path', 'text', 'duration']

MISSING VALUES:
audio_path    0
text          0
duration      0
dtype: int64

DUPLICATES:
Duplicate audio paths: 0
Duplicate texts: 2

DURATION STATISTICS:
count    3708.000000
mean        6.638374
std         2.966617
min         2.006562
25%         3.924187
50%         6.600000
75%         8.920000
max        19.900000
Name: duration, dtype: float64


In [4]:
bins = [0, 1, 2, 5, 10, 20, 30, float("inf")]

labels = [
    "0-1 sec",
    "1-2 sec",
    "2-5 sec",
    "5-10 sec",
    "10-20 sec",
    "20-30 sec",
    "30+ sec"
]

df["duration_group"] = pd.cut(
    df["duration"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

duration_stats = (
    df["duration_group"]
    .value_counts()
    .sort_index()
)

print(duration_stats)

print("\nTotal duration (hours):")
print(df["duration"].sum() / 3600)

duration_group
0-1 sec         0
1-2 sec         0
2-5 sec      1292
5-10 sec     1943
10-20 sec     473
20-30 sec       0
30+ sec         0
Name: count, dtype: int64

Total duration (hours):
6.837525190972222


In [5]:
import soundfile as sf
import pandas as pd
from tqdm.auto import tqdm

audio_results = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    audio_path = DATASET_DIR / row["audio_path"]

    result = {
        "audio_path": row["audio_path"],
        "exists": audio_path.is_file(),
        "sample_rate": None,
        "channels": None,
        "actual_duration": None,
        "error": None
    }

    if result["exists"]:
        try:
            info = sf.info(str(audio_path))

            result["sample_rate"] = info.samplerate
            result["channels"] = info.channels
            result["actual_duration"] = info.duration

        except Exception as e:
            result["error"] = str(e)

    audio_results.append(result)

audio_df = pd.DataFrame(audio_results)

print("=" * 45)
print("AUDIO VALIDATION REPORT")
print("=" * 45)

print("Total:", len(audio_df))
print("Missing audio:", (~audio_df["exists"]).sum())
print("Corrupted/unreadable:", audio_df["error"].notna().sum())

print("\nSAMPLE RATES:")
print(audio_df["sample_rate"].value_counts(dropna=False))

print("\nCHANNELS:")
print(audio_df["channels"].value_counts(dropna=False))

  0%|          | 0/3708 [00:00<?, ?it/s]

AUDIO VALIDATION REPORT
Total: 3708
Missing audio: 0
Corrupted/unreadable: 0

SAMPLE RATES:
sample_rate
16000    3708
Name: count, dtype: int64

CHANNELS:
channels
1    3708
Name: count, dtype: int64


In [6]:
df_checked = df.merge(
    audio_df,
    on="audio_path",
    how="left",
    validate="one_to_one"
)

df_checked["duration_diff"] = abs(
    df_checked["duration"] -
    df_checked["actual_duration"]
)

print("Duration mismatch > 0.1 sec:",
      (df_checked["duration_diff"] > 0.1).sum())

print("Duration mismatch > 1 sec:",
      (df_checked["duration_diff"] > 1).sum())

display(
    df_checked.sort_values(
        "duration_diff",
        ascending=False
    ).head(10)
)

Duration mismatch > 0.1 sec: 0
Duration mismatch > 1 sec: 0


,audio_path,text,duration,duration_group,exists,sample_rate,channels,actual_duration,error,duration_diff
0,train/audio/hf_audio_000001.wav,kún uyasına sıymay dolanıp qıp qızıl shoqtay j...,6.95225,5-10 sec,True,16000,1,6.95225,None,0.0
2463,train/audio/mendeley_audio_000458.wav,ele tabılmay atırǵanın śala esitse de anıǵın s...,7.02000,5-10 sec,True,16000,1,7.02000,None,0.0
2465,train/audio/mendeley_audio_000460.wav,assalawma áleykumin esitip júregi biraz toqtas...,6.16200,5-10 sec,True,16000,1,6.16200,None,0.0
2466,train/audio/mendeley_audio_000461.wav,yamasa basqa bir tillerdiń sózlerin hám olardı...,7.02000,5-10 sec,True,16000,1,7.02000,None,0.0
2467,train/audio/mendeley_audio_000462.wav,hám aytıw uśın túsindirme sózlik ádebiy tildiń...,6.13600,5-10 sec,True,16000,1,6.13600,None,0.0
2468,train/audio/mendeley_audio_000463.wav,qollanıladı eki tillik yamasa kóp tillik awdar...,5.98000,5-10 sec,True,16000,1,5.98000,None,0.0
2469,train/audio/mendeley_audio_000464.wav,sózliklerden basqa da terminologiyalıq dialekt...,7.64400,5-10 sec,True,16000,1,7.64400,None,0.0
2470,train/audio/mendeley_audio_000466.wav,tildegi jazba hám awızśa qollanılatuǵın sózler...,6.31800,5-10 sec,True,16000,1,6.31800,None,0.0
2471,train/audio/mendeley_audio_000467.wav,qollanılatuǵın tiykarǵı sózlerdiń tuwra hám aw...,6.65600,5-10 sec,True,16000,1,6.65600,None,0.0
2472,train/audio/mendeley_audio_000469.wav,ádebiy til uśın qabıl etilgen norması sońǵı tú...,6.70800,5-10 sec,True,16000,1,6.70800,None,0.0


In [7]:
duplicates = df_checked[
    df_checked["text"].duplicated(keep=False)
]

display(
    duplicates[
        ["audio_path", "text", "duration"]
    ]
)

,audio_path,text,duration
771,train/audio/hf_audio_000972.wav,qayırlı kún dedi kishkene shahzada,2.217500
1139,train/audio/hf_audio_001417.wav,qayırlı kún dedi kishkene shahzada,2.043375
1158,train/audio/hf_audio_001439.wav,qayırlı kún dedi kishkene shahzada,2.089812


In [8]:
df_checked["source"] = df_checked["audio_path"].apply(
    lambda x: (
        "Mendeley" if "mendeley_audio" in x
        else "HuggingFace" if "hf_audio" in x
        else "Unknown"
    )
)

source_stats = df_checked.groupby("source").agg(
    samples=("audio_path", "count"),
    total_hours=("actual_duration", lambda x: x.sum() / 3600),
    min_duration=("actual_duration", "min"),
    max_duration=("actual_duration", "max"),
    mean_duration=("actual_duration", "mean")
)

display(source_stats)

,samples,total_hours,min_duration,max_duration,mean_duration
source,,,,,
HuggingFace,2090,2.769272,2.006562,11.643562,4.770039
Mendeley,1618,4.068253,4.080000,19.900000,9.051737


In [9]:
mendeley_df = df_checked[
    df_checked["audio_path"].str.contains(
        "mendeley_audio",
        na=False
    )
].copy()

print("Mendeley samples:", len(mendeley_df))

print(
    "Mendeley duration (hours):",
    mendeley_df["actual_duration"].sum() / 3600
)

display(mendeley_df.head())

Mendeley samples: 1618
Mendeley duration (hours): 4.0682527777777775


,audio_path,text,duration,duration_group,exists,sample_rate,channels,actual_duration,error,duration_diff,source
2090,train/audio/mendeley_audio_000002.wav,ústinde turıp basın ana jaqqa bir mına jaqqa b...,8.684,5-10 sec,True,16000,1,8.684,None,0.0,Mendeley
2091,train/audio/mendeley_audio_000003.wav,dógereginde hámme jerde órip júr hám arasında ...,8.164,5-10 sec,True,16000,1,8.164,None,0.0,Mendeley
2092,train/audio/mendeley_audio_000004.wav,oynaydı hámme qosıq aytadı quslar sayraydı jaw...,9.724,5-10 sec,True,16000,1,9.724,None,0.0,Mendeley
2093,train/audio/mendeley_audio_000005.wav,titirkendirip jibergeni menen de báribir oǵan ...,8.788,5-10 sec,True,16000,1,8.788,None,0.0,Mendeley
2094,train/audio/mendeley_audio_000006.wav,śaǵılısqan gúmis tamśılar marjan monśaqlar kib...,8.450,5-10 sec,True,16000,1,8.450,None,0.0,Mendeley


In [10]:
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

mendeley_df = df_checked[
    (df_checked["source"] == "Mendeley") &
    (df_checked["exists"]) &
    (df_checked["error"].isna()) &
    (df_checked["text"].notna()) &
    (df_checked["text"].astype(str).str.strip() != "") &
    (df_checked["actual_duration"] >= 2) &
    (df_checked["actual_duration"] <= 30)
].copy()

print("Selected samples:", len(mendeley_df))

print(
    "Total hours:",
    round(mendeley_df["actual_duration"].sum() / 3600, 3)
)

OUTPUT_PATH = RESULTS_DIR / "mendeley_evaluation_audio_manifest.csv"

mendeley_df[
    ["audio_path", "text", "actual_duration"]
].to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

Selected samples: 1618
Total hours: 4.068
Saved: /home/intern/projects/karakalpak_stt/results/mendeley_external_eval.csv


In [11]:
import pandas as pd
import shutil
from pathlib import Path
from tqdm.auto import tqdm

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

DATASET_DIR = (
    PROJECT_DIR
    / "external_validation"
    / "karakalpak_combined"
)

OUTPUT_DIR = PROJECT_DIR / "external_validation"

AUDIO_OUTPUT_DIR = OUTPUT_DIR / "Mendeley_train_audio"

AUDIO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Original dataset
df = pd.read_csv(DATASET_DIR / "train.csv")

# Faqat Mendeley
mendeley_df = df[
    df["audio_path"].str.contains(
        "mendeley_audio",
        na=False
    )
].copy()

print("Mendeley samples:", len(mendeley_df))

# Audio fayllarni nusxalash
new_audio_paths = []

for _, row in tqdm(
    mendeley_df.iterrows(),
    total=len(mendeley_df)
):

    source = DATASET_DIR / row["audio_path"]

    destination = (
        AUDIO_OUTPUT_DIR / source.name
    )

    if not source.is_file():
        raise FileNotFoundError(source)

    shutil.copy2(
        source,
        destination
    )

    new_audio_paths.append(
        f"Mendeley_train_audio/{source.name}"
    )

# CSV audio yo'llarini yangilaymiz
mendeley_df["audio_path"] = new_audio_paths

# Faqat kerakli ustunlar
mendeley_df = mendeley_df[
    ["audio_path", "text", "duration"]
]

# CSV saqlash
CSV_OUTPUT_PATH = OUTPUT_DIR / "Mendeley_train.csv"

mendeley_df.to_csv(
    CSV_OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("\n✅ Mendeley dataset separated!")

print("CSV:", CSV_OUTPUT_PATH)
print("Audio folder:", AUDIO_OUTPUT_DIR)

print("Total samples:", len(mendeley_df))

Mendeley samples: 1618


  0%|          | 0/1618 [00:00<?, ?it/s]


✅ Mendeley dataset separated!
CSV: /home/intern/projects/karakalpak_stt/external_validation/Mendeley_train.csv
Audio folder: /home/intern/projects/karakalpak_stt/external_validation/Mendeley_train_audio
Total samples: 1618


In [13]:
mendeley_check = pd.read_csv(
    OUTPUT_DIR / "Mendeley_train.csv"
)

audio_files = list(
    AUDIO_OUTPUT_DIR.glob("*.wav")
)

missing_audio = [
    path
    for path in mendeley_check["audio_path"]
    if not (OUTPUT_DIR / path).is_file()
]

print("=" * 45)
print("MENDELEY DATASET VERIFICATION")
print("=" * 45)

print("CSV rows:", len(mendeley_check))
print("Audio files:", len(audio_files))
print("Missing audio:", len(missing_audio))

print(
    "Total hours:",
    round(mendeley_check["duration"].sum() / 3600, 3)
)

display(mendeley_check.head())

MENDELEY DATASET VERIFICATION
CSV rows: 1618
Audio files: 1618
Missing audio: 0
Total hours: 4.068


,audio_path,text,duration
0,Mendeley_train_audio/mendeley_audio_000002.wav,ústinde turıp basın ana jaqqa bir mına jaqqa b...,8.684
1,Mendeley_train_audio/mendeley_audio_000003.wav,dógereginde hámme jerde órip júr hám arasında ...,8.164
2,Mendeley_train_audio/mendeley_audio_000004.wav,oynaydı hámme qosıq aytadı quslar sayraydı jaw...,9.724
3,Mendeley_train_audio/mendeley_audio_000005.wav,titirkendirip jibergeni menen de báribir oǵan ...,8.788
4,Mendeley_train_audio/mendeley_audio_000006.wav,śaǵılısqan gúmis tamśılar marjan monśaqlar kib...,8.450


In [14]:
from pathlib import Path
import json

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

MODEL_DIR = PROJECT_DIR / "models" / "karakalpak_stt_optimized"

print("MODEL FILES:\n")

for path in MODEL_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(MODEL_DIR))

print("\n" + "=" * 50)

# Adapter konfiguratsiyasi
config_path = MODEL_DIR / "adapter" / "adapter_config.json"

if config_path.exists():
    with open(config_path, encoding="utf-8") as f:
        config = json.load(f)

    print("BASE MODEL:", config.get("base_model_name_or_path"))
    print("LORA RANK:", config.get("r"))
    print("LORA ALPHA:", config.get("lora_alpha"))
    print("TARGET MODULES:", config.get("target_modules"))
else:
    print("Adapter config not found!")

MODEL FILES:

decoder_config.json
kenlm/karakalpak_5gram.arpa
adapter/README.md
adapter/adapter_config.json
adapter/adapter_model.safetensors
adapter/model_metadata.json
adapter/processor/tokenizer_config.json
adapter/processor/processor_config.json
adapter/processor/vocab.json
processor/tokenizer_config.json
processor/processor_config.json
processor/vocab.json

BASE MODEL: facebook/wav2vec2-xls-r-300m
LORA RANK: 8
LORA ALPHA: 16
TARGET MODULES: ['v_proj', 'q_proj']


In [15]:
import pandas as pd

EVAL_DIR = PROJECT_DIR / "external_validation"

CSV_PATH = EVAL_DIR / "Mendeley_train.csv"

eval_df = pd.read_csv(CSV_PATH)

eval_df["full_audio_path"] = eval_df["audio_path"].apply(
    lambda x: str(EVAL_DIR / x)
)

print("Evaluation samples:", len(eval_df))

print(
    "Missing audio:",
    sum(not Path(p).is_file() for p in eval_df["full_audio_path"])
)

display(eval_df.head())

Evaluation samples: 1618
Missing audio: 0


,audio_path,text,duration,full_audio_path
0,Mendeley_train_audio/mendeley_audio_000002.wav,ústinde turıp basın ana jaqqa bir mına jaqqa b...,8.684,/home/intern/projects/karakalpak_stt/external_...
1,Mendeley_train_audio/mendeley_audio_000003.wav,dógereginde hámme jerde órip júr hám arasında ...,8.164,/home/intern/projects/karakalpak_stt/external_...
2,Mendeley_train_audio/mendeley_audio_000004.wav,oynaydı hámme qosıq aytadı quslar sayraydı jaw...,9.724,/home/intern/projects/karakalpak_stt/external_...
3,Mendeley_train_audio/mendeley_audio_000005.wav,titirkendirip jibergeni menen de báribir oǵan ...,8.788,/home/intern/projects/karakalpak_stt/external_...
4,Mendeley_train_audio/mendeley_audio_000006.wav,śaǵılısqan gúmis tamśılar marjan monśaqlar kib...,8.450,/home/intern/projects/karakalpak_stt/external_...


In [17]:
from safetensors import safe_open
from pathlib import Path

MODEL_DIR = Path(
    "/home/intern/projects/karakalpak_stt/"
    "models/karakalpak_stt_optimized"
)

ADAPTER_PATH = (
    MODEL_DIR / "adapter" / "adapter_model.safetensors"
)

with safe_open(str(ADAPTER_PATH), framework="pt", device="cpu") as f:
    keys = list(f.keys())

print("Total tensors:", len(keys))

print("\nCTC HEAD / LM HEAD:")
for key in keys:
    if "lm_head" in key or "ctc" in key.lower():
        print(key)

print("\nFIRST 15 TENSORS:")
for key in keys[:15]:
    print(key)

Total tensors: 98

CTC HEAD / LM HEAD:
base_model.model.lm_head.bias
base_model.model.lm_head.weight

FIRST 15 TENSORS:
base_model.model.lm_head.bias
base_model.model.lm_head.weight
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_A.weight
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_B.weight
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_A.weight
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_B.weight
base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_A.weight
base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_B.weight
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_A.weight
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_B.weight
base_model.model.wav2vec2.encoder.layers.10.attention.q_proj.lora_A.weight
base_model.model.wav2vec2.encoder.layers.10.attention.q_proj.lora_B.weight
base_model.model.wav2vec2.encoder.layers.10.attention.v_proj.lora_A.weight
b

In [18]:
import json

metadata_path = (
    MODEL_DIR / "adapter" / "model_metadata.json"
)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(json.dumps(
    metadata,
    indent=2,
    ensure_ascii=False
))

{
  "base_model": "facebook/wav2vec2-xls-r-300m",
  "checkpoint_step": 13023,
  "vocab_size": 36,
  "lora_rank": 8,
  "lora_alpha": 16,
  "validation_wer_greedy": 50.0032,
  "validation_cer_greedy": 11.5323,
  "validation_wer_kenlm": 27.0632,
  "validation_cer_kenlm": 10.0256
}


In [19]:
decoder_path = MODEL_DIR / "decoder_config.json"

with open(decoder_path, "r", encoding="utf-8") as f:
    decoder_config = json.load(f)

print(json.dumps(
    decoder_config,
    indent=2,
    ensure_ascii=False
))

{
  "base_model": "facebook/wav2vec2-xls-r-300m",
  "alpha": 0.65,
  "beta": 1.5,
  "beam_width": 200,
  "sample_rate": 16000,
  "test_wer": 21.5938,
  "test_cer": 7.789
}


In [20]:
from pathlib import Path
import json

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
    / "full_lora_step_8500"
)

print("CHECKPOINT EXISTS:", CHECKPOINT_DIR.exists())

print("\nCHECKPOINT FILES:")
for file in CHECKPOINT_DIR.rglob("*"):
    if file.is_file():
        print(
            file.relative_to(CHECKPOINT_DIR),
            "|",
            round(file.stat().st_size / (1024**2), 2),
            "MB"
        )

print("\nJSON CONFIGURATIONS:")

for file in CHECKPOINT_DIR.rglob("*.json"):
    print(f"\n--- {file.name} ---")

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

CHECKPOINT EXISTS: True

CHECKPOINT FILES:
adapter_config.json | 0.0 MB
tokenizer_config.json | 0.0 MB
processor_config.json | 0.0 MB
vocab.json | 0.0 MB
training_state.pt | 10.13 MB

JSON CONFIGURATIONS:

--- adapter_config.json ---
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "facebook/wav2vec2-xls-r-300m",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": false,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.0,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": [
    "lm_head"
  ],
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_vers

In [21]:
import pandas as pd

EVAL_DIR = PROJECT_DIR / "external_validation"

CSV_PATH = EVAL_DIR / "Mendeley_train.csv"

eval_df = pd.read_csv(CSV_PATH)

eval_df["full_audio_path"] = eval_df["audio_path"].apply(
    lambda x: str(EVAL_DIR / x)
)

print("Samples:", len(eval_df))
print("Total hours:", eval_df["duration"].sum() / 3600)

print(
    "Missing audio:",
    sum(
        not Path(p).is_file()
        for p in eval_df["full_audio_path"]
    )
)

display(eval_df.head())

Samples: 1618
Total hours: 4.0682527777777775
Missing audio: 0


,audio_path,text,duration,full_audio_path
0,Mendeley_train_audio/mendeley_audio_000002.wav,ústinde turıp basın ana jaqqa bir mına jaqqa b...,8.684,/home/intern/projects/karakalpak_stt/external_...
1,Mendeley_train_audio/mendeley_audio_000003.wav,dógereginde hámme jerde órip júr hám arasında ...,8.164,/home/intern/projects/karakalpak_stt/external_...
2,Mendeley_train_audio/mendeley_audio_000004.wav,oynaydı hámme qosıq aytadı quslar sayraydı jaw...,9.724,/home/intern/projects/karakalpak_stt/external_...
3,Mendeley_train_audio/mendeley_audio_000005.wav,titirkendirip jibergeni menen de báribir oǵan ...,8.788,/home/intern/projects/karakalpak_stt/external_...
4,Mendeley_train_audio/mendeley_audio_000006.wav,śaǵılısqan gúmis tamśılar marjan monśaqlar kib...,8.450,/home/intern/projects/karakalpak_stt/external_...


In [22]:
import torch
from pathlib import Path

CHECKPOINT_DIR = Path(
    "/home/intern/projects/karakalpak_stt/"
    "checkpoints/full_lora_step_8500"
)

STATE_PATH = CHECKPOINT_DIR / "training_state.pt"

# Faqat o'zimiz yaratgan ishonchli checkpointni yuklaymiz
checkpoint = torch.load(
    STATE_PATH,
    map_location="cpu",
    weights_only=True
)

print("=" * 50)
print("CHECKPOINT INFORMATION")
print("=" * 50)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nTOP LEVEL KEYS:")

    for key, value in checkpoint.items():
        print(
            f"{key}: {type(value).__name__}"
        )

CHECKPOINT INFORMATION
Checkpoint type: <class 'dict'>

TOP LEVEL KEYS:
global_step: int
model_weights: dict
optimizer_state: dict
scheduler_state: dict
training_history: list
validation_history: list
base_model_name: str
vocab_size: int
grad_accum_steps: int
torch_rng_state: Tensor
cuda_rng_state: list


In [23]:
state = checkpoint.get(
    "model_state_dict",
    checkpoint.get("model_state", {})
)

if isinstance(state, dict):

    print("Total tensors:", len(state))

    print("\nCTC HEAD WEIGHTS:")

    for key in state:
        if "lm_head" in key:
            print(key)

    print("\nLORA WEIGHTS:")

    lora_keys = [
        key for key in state
        if "lora_" in key.lower()
    ]

    print("LoRA tensors:", len(lora_keys))

    for key in lora_keys[:10]:
        print(key)

Total tensors: 0

CTC HEAD WEIGHTS:

LORA WEIGHTS:
LoRA tensors: 0


In [ ]:
state = checkpoint["model_weights"]

print("=" * 50)
print("MODEL WEIGHTS INFORMATION")
print("=" * 50)

print("Global step:", checkpoint["global_step"])
print("Base model:", checkpoint["base_model_name"])
print("Vocabulary size:", checkpoint["vocab_size"])

print("\nTotal tensors:", len(state))

print("\nCTC HEAD WEIGHTS:")
for key in state:
    if "lm_head" in key:
        print(key, state[key].shape)

print("\nLORA WEIGHTS:")

lora_keys = [
    key for key in state
    if "lora_" in key.lower()
]

print("LoRA tensors:", len(lora_keys))

for key in lora_keys[:10]:
    print(key, state[key].shape)

MODEL WEIGHTS INFORMATION
Global step: 8500
Base model: facebook/wav2vec2-xls-r-300m
Vocabulary size: 36

Total tensors: 98

CTC HEAD WEIGHTS:
base_model.model.lm_head.modules_to_save.default.weight torch.Size([36, 1024])
base_model.model.lm_head.modules_to_save.default.bias torch.Size([36])

LORA WEIGHTS:
LoRA tensors: 96
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_A.default.weight torch.Size([8, 1024])
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_B.default.weight torch.Size([1024, 8])
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_A.default.weight torch.Size([8, 1024])
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_B.default.weight torch.Size([1024, 8])
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_A.default.weight torch.Size([8, 1024])
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_B.default.weight torch.Size([1024, 8])
base_model.model.wav2vec2.encoder.layers.1.attentio

In [25]:
import json

config_path = CHECKPOINT_DIR / "adapter_config.json"

with open(config_path, "r", encoding="utf-8") as f:
    adapter_config = json.load(f)

print("LoRA rank:", adapter_config["r"])
print("LoRA alpha:", adapter_config["lora_alpha"])
print("Target modules:", adapter_config["target_modules"])
print("Modules to save:", adapter_config.get("modules_to_save"))

LoRA rank: 8
LoRA alpha: 16
Target modules: ['v_proj', 'q_proj']
Modules to save: ['lm_head']


In [26]:
state = checkpoint["model_weights"]

print("Global step:", checkpoint["global_step"])
print("Total saved tensors:", len(state))

print("\nCTC HEAD:")
for key in state:
    if "lm_head" in key:
        print(key, state[key].shape)

print("\nFIRST 10 WEIGHT NAMES:")
for key in list(state.keys())[:10]:
    print(key)

Global step: 8500
Total saved tensors: 98

CTC HEAD:
base_model.model.lm_head.modules_to_save.default.weight torch.Size([36, 1024])
base_model.model.lm_head.modules_to_save.default.bias torch.Size([36])

FIRST 10 WEIGHT NAMES:
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_A.default.weight
base_model.model.wav2vec2.encoder.layers.0.attention.v_proj.lora_B.default.weight
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_A.default.weight
base_model.model.wav2vec2.encoder.layers.0.attention.q_proj.lora_B.default.weight
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_A.default.weight
base_model.model.wav2vec2.encoder.layers.1.attention.v_proj.lora_B.default.weight
base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_A.default.weight
base_model.model.wav2vec2.encoder.layers.1.attention.q_proj.lora_B.default.weight
base_model.model.wav2vec2.encoder.layers.2.attention.v_proj.lora_A.default.weight
base_model.model.wav2vec2.encoder.l

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(
    str(CHECKPOINT_DIR)
)

base_model = Wav2Vec2ForCTC.from_pretrained(
    checkpoint["base_model_name"],
    vocab_size=checkpoint["vocab_size"],
    ignore_mismatched_sizes=True
)

lora_config = LoraConfig(
    r=adapter_config["r"],
    lora_alpha=adapter_config["lora_alpha"],
    target_modules=adapter_config["target_modules"],
    modules_to_save=adapter_config["modules_to_save"],
    lora_dropout=adapter_config.get("lora_dropout", 0.0),
    bias="none"
)

model = get_peft_model(
    base_model,
    lora_config
)

print("Model architecture created!")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model architecture created!


In [ ]:

state = checkpoint["model_weights"]


load_result = model.load_state_dict(
    state,
    strict=False
)

print("Missing keys:", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))

print("\nMISSING:")
print(load_result.missing_keys[:20])

print("\nUNEXPECTED:")
print(load_result.unexpected_keys[:20])

Missing keys: 424
Unexpected keys: 0

MISSING:
['base_model.model.wav2vec2.masked_spec_embed', 'base_model.model.wav2vec2.feature_extractor.conv_layers.0.conv.weight', 'base_model.model.wav2vec2.feature_extractor.conv_layers.0.conv.bias', 'base_model.model.wav2vec2.feature_extractor.conv_layers.0.layer_norm.weight', 'base_model.model.wav2vec2.feature_extractor.conv_layers.0.layer_norm.bias', 'base_model.model.wav2vec2.feature_extractor.conv_layers.1.conv.weight', 'base_model.model.wav2vec2.feature_extractor.conv_layers.1.conv.bias', 'base_model.model.wav2vec2.feature_extractor.conv_layers.1.layer_norm.weight', 'base_model.model.wav2vec2.feature_extractor.conv_layers.1.layer_norm.bias', 'base_model.model.wav2vec2.feature_extractor.conv_layers.2.conv.weight', 'base_model.model.wav2vec2.feature_extractor.conv_layers.2.conv.bias', 'base_model.model.wav2vec2.feature_extractor.conv_layers.2.layer_norm.weight', 'base_model.model.wav2vec2.feature_extractor.conv_layers.2.layer_norm.bias', 'base

In [29]:
model_keys = set(model.state_dict().keys())
saved_keys = set(state.keys())

missing = model_keys - saved_keys
unexpected = saved_keys - model_keys

print("Model tensors:", len(model_keys))
print("Checkpoint tensors:", len(saved_keys))

print("Missing:", len(missing))
print("Unexpected:", len(unexpected))

print("\nMissing LoRA/head:")
for key in sorted(missing):
    if "lora_" in key or "lm_head" in key:
        print(key)

print("\nUnexpected LoRA/head:")
for key in sorted(unexpected):
    if "lora_" in key or "lm_head" in key:
        print(key)

Model tensors: 522
Checkpoint tensors: 98
Missing: 424
Unexpected: 0

Missing LoRA/head:
base_model.model.lm_head.original_module.bias
base_model.model.lm_head.original_module.weight

Unexpected LoRA/head:


In [30]:
import torch

# Checkpointdagi barcha tensor nomlari
saved_keys = set(state.keys())

# Modeldagi tensor nomlari
model_keys = set(model.state_dict().keys())

# LoRA va CTC head uchun kerakli tensorlar
required_keys = {
    key for key in model_keys
    if "lora_" in key
    or "modules_to_save.default" in key
}

missing_required = required_keys - saved_keys

print("=" * 50)
print("MODEL WEIGHTS VERIFICATION")
print("=" * 50)

print("Saved tensors:", len(saved_keys))
print("Required LoRA/Head tensors:", len(required_keys))
print("Missing required tensors:", len(missing_required))

if missing_required:
    for key in sorted(missing_required):
        print("MISSING:", key)
else:
    print("✅ All required LoRA and CTC head tensors are present!")

# Haqiqiy yuklangan qiymatlarni solishtiramiz
loaded_state = model.state_dict()

mismatched = []

for key in required_keys & saved_keys:

    if not torch.equal(
        loaded_state[key].detach().cpu(),
        state[key].detach().cpu()
    ):
        mismatched.append(key)

print("Mismatched tensor values:", len(mismatched))

if not missing_required and not mismatched:
    print("✅ 8500-step trainable weights successfully restored!")
else:
    print("❌ Weight verification failed!")

MODEL WEIGHTS VERIFICATION
Saved tensors: 98
Required LoRA/Head tensors: 98
Missing required tensors: 0
✅ All required LoRA and CTC head tensors are present!
Mismatched tensor values: 0
✅ 8500-step trainable weights successfully restored!


In [31]:
assert not missing_required, "Required weights are missing!"
assert not mismatched, "Loaded weights do not match!"

model = model.to(device)
model.eval()

print("✅ Model ready for inference!")
print("Device:", device)
print("Checkpoint step:", checkpoint["global_step"])

✅ Model ready for inference!
Device: cuda
Checkpoint step: 8500


In [ ]:
import torch
import pandas as pd
import soundfile as sf
import numpy as np

from pathlib import Path
from jiwer import wer, cer

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")
EVAL_DIR = PROJECT_DIR / "external_validation"

eval_df = pd.read_csv(EVAL_DIR / "Mendeley_train.csv")

print("Total samples:", len(eval_df))

def transcribe_audio(audio_path):
    audio, sr = sf.read(str(audio_path), dtype="float32")

    assert sr == 16000, f"Unexpected sample rate: {sr}"
    assert audio.ndim == 1, "Audio must be mono"

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(device)

    with torch.inference_mode():
        logits = model(
            input_values=input_values
        ).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(
        predicted_ids
    )[0]

    return prediction

Total samples: 1618


In [33]:
for i in range(5):

    row = eval_df.iloc[i]

    audio_path = EVAL_DIR / row["audio_path"]

    prediction = transcribe_audio(audio_path)

    print("=" * 70)
    print(f"AUDIO {i+1}")
    print("=" * 70)

    print("REFERENCE:")
    print(row["text"])

    print("\nPREDICTION:")
    print(prediction)

    print("\nWER:", round(wer(
        str(row["text"]).lower(),
        prediction.lower()
    ) * 100, 2), "%")

    print("\nCER:", round(cer(
        str(row["text"]).lower(),
        prediction.lower()
    ) * 100, 2), "%")

AUDIO 1
REFERENCE:
ústinde turıp basın ana jaqqa bir mına jaqqa bir burıp tula bedenin tarap quyaśqa qızdırınıp atır al tawıqlar śe olar úydiń

PREDICTION:
ústinde turıpbasın ana jaqqa br mınajaqqa bir burip tula bedenin tarap qoyashqa qızdırınıp atır al tawıqlar she olar úydıń

WER: 42.86 %

CER: 8.13 %
AUDIO 2
REFERENCE:
dógereginde hámme jerde órip júr hám arasında śaqırıp qoyadı báhár kelse qanday kewilli boladı deseś hámme kewillenedi hámme

PREDICTION:
dógereginde hámmi jeti órip júr hám arasında shaqırıp qoyadı bahar kelsi qanday kewle boladı desesh hámme kewllenedi hámme

WER: 44.44 %

CER: 12.1 %
AUDIO 3
REFERENCE:
oynaydı hámme qosıq aytadı quslar sayraydı jawın ele silpilep jawıp turǵan edi sapsalqın tamśılar bahadırdıń betine tiyip

PREDICTION:
oynayda hámmin qosı aytadı quslar sayraydı jawın ele sil pilep jawap turǵan edi sap salqın tamshılar bahardıń betinne tiyip

WER: 64.71 %

CER: 9.92 %
AUDIO 4
REFERENCE:
titirkendirip jibergeni menen de báribir oǵan bul júdá jaǵımlı 

In [34]:
from tqdm.auto import tqdm
import time

references = []
predictions = []
results = []

start_time = time.time()

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Mendeley Evaluation"
):

    audio_path = EVAL_DIR / row["audio_path"]

    reference = str(row["text"]).lower().strip()

    prediction = transcribe_audio(audio_path).lower().strip()

    references.append(reference)
    predictions.append(prediction)

    results.append({
        "audio_path": row["audio_path"],
        "duration": row["duration"],
        "reference": reference,
        "prediction": prediction,
        "wer": wer(reference, prediction) * 100,
        "cer": cer(reference, prediction) * 100
    })

evaluation_seconds = time.time() - start_time

total_wer = wer(references, predictions) * 100
total_cer = cer(references, predictions) * 100

print("=" * 50)
print("MENDELEY EXTERNAL EVALUATION")
print("=" * 50)

print(f"Checkpoint: {checkpoint['global_step']}")
print(f"Samples: {len(references)}")
print(f"Greedy WER: {total_wer:.4f}%")
print(f"Greedy CER: {total_cer:.4f}%")
print(f"Evaluation time: {evaluation_seconds:.2f} seconds")

Mendeley Evaluation:   0%|          | 0/1618 [00:00<?, ?it/s]

MENDELEY EXTERNAL EVALUATION
Checkpoint: 8500
Samples: 1618
Greedy WER: 59.8314%
Greedy CER: 13.4100%
Evaluation time: 157.40 seconds


In [35]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert len(results) == len(eval_df) == 1618, "Evaluation to‘liq tugamagan!"

# Har bir audioning natijasi
results_df = pd.DataFrame(results)

results_df.to_csv(
    RESULTS_DIR / "mendeley_step8500_greedy_predictions.csv",
    index=False,
    encoding="utf-8"
)

# Umumiy natijalar
metrics_df = pd.DataFrame([{
    "checkpoint": 8500,
    "dataset": "Mendeley",
    "samples": len(references),
    "greedy_wer": total_wer,
    "greedy_cer": total_cer,
    "evaluation_seconds": evaluation_seconds
}])

metrics_df.to_csv(
    RESULTS_DIR / "mendeley_step8500_greedy_metrics.csv",
    index=False
)

print("✅ Results saved!")
print(f"WER: {total_wer:.4f}%")
print(f"CER: {total_cer:.4f}%")

✅ Results saved!
WER: 59.8314%
CER: 13.4100%


In [ ]:
from pyctcdecode import build_ctcdecoder
import json

KENLM_PATH = (
    PROJECT_DIR
    / "kenlm"
    / "karakalpak_5gram.binary"
)

vocab = processor.tokenizer.get_vocab()

labels = [
    token
    for token, idx in sorted(
        vocab.items(),
        key=lambda x: x[1]
    )
]

labels = [
    " " if token == "|" else token
    for token in labels
]

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(KENLM_PATH),
    alpha=0.65,
    beta=1.5
)

print("✅ KenLM decoder loaded!")
print("Vocabulary size:", len(labels))

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
No known unigrams provided, decoding results might be a lot worse.


✅ KenLM decoder loaded!
Vocabulary size: 36


In [37]:
import soundfile as sf
import numpy as np
import torch

row = eval_df.iloc[0]

audio_path = EVAL_DIR / row["audio_path"]

audio, sr = sf.read(
    str(audio_path),
    dtype="float32"
)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

with torch.inference_mode():
    logits = model(
        input_values=inputs.input_values.to(device)
    ).logits

logits_np = logits[0].float().cpu().numpy()

prediction_kenlm = decoder.decode(
    logits_np,
    beam_width=200
)

print("REFERENCE:")
print(row["text"])

print("\nGREEDY:")
print(predictions[0])

print("\nKENLM:")
print(prediction_kenlm)

REFERENCE:
ústinde turıp basın ana jaqqa bir mına jaqqa bir burıp tula bedenin tarap quyaśqa qızdırınıp atır al tawıqlar śe olar úydiń

GREEDY:
ústinde turıpbasın ana jaqqa br mınajaqqa bir burip tula bedenin tarap qoyashqa qızdırınıp atır al tawıqlar she olar úydıń

KENLM:
ústinde turıpbasınanajaqqa bir mınajaqqabirburip tulabedenintarapqoyasqa qızdırınıpatıraltawıqlar she olar úydiń


In [38]:
print("=" * 50)
print("TOKENIZER CONFIGURATION")
print("=" * 50)

print("Vocabulary size:", len(processor.tokenizer))
print("PAD token:", processor.tokenizer.pad_token)
print("PAD token ID:", processor.tokenizer.pad_token_id)

print("UNK token:", processor.tokenizer.unk_token)
print("UNK token ID:", processor.tokenizer.unk_token_id)

print(
    "Word delimiter:",
    processor.tokenizer.word_delimiter_token
)

print("\nVOCABULARY:")

vocab = processor.tokenizer.get_vocab()

for token, idx in sorted(
    vocab.items(),
    key=lambda x: x[1]
):
    print(idx, repr(token))

TOKENIZER CONFIGURATION
Vocabulary size: 36
PAD token: [PAD]
PAD token ID: 35
UNK token: [UNK]
UNK token ID: 34
Word delimiter: |

VOCABULARY:
0 'a'
1 'b'
2 'c'
3 'd'
4 'e'
5 'f'
6 'g'
7 'h'
8 'i'
9 'j'
10 'k'
11 'l'
12 'm'
13 'n'
14 'o'
15 'p'
16 'q'
17 'r'
18 's'
19 't'
20 'u'
21 'v'
22 'w'
23 'x'
24 'y'
25 'z'
26 'á'
27 'í'
28 'ó'
29 'ú'
30 'ı'
31 'ń'
32 'ǵ'
33 '|'
34 '[UNK]'
35 '[PAD]'


In [39]:
vocab = processor.tokenizer.get_vocab()

labels = [
    token
    for token, idx in sorted(
        vocab.items(),
        key=lambda x: x[1]
    )
]

labels = [
    " " if token == "|" else token
    for token in labels
]

print("Labels:", labels)

print("\nBlank token ID:")
print(processor.tokenizer.pad_token_id)

print("\nSpace token ID:")
print(vocab["|"])

print("\nDecoder alphabet:")
print(decoder._alphabet.labels)

print("\nLabels count:", len(labels))
print("Decoder alphabet count:", len(decoder._alphabet.labels))

Labels: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'í', 'ó', 'ú', 'ı', 'ń', 'ǵ', ' ', '[UNK]', '[PAD]']

Blank token ID:
35

Space token ID:
33

Decoder alphabet:
['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'í', 'ó', 'ú', 'ı', 'ń', 'ǵ', ' ', '⁇', '']

Labels count: 36
Decoder alphabet count: 36


In [40]:
import kenlm

LM_PATH = PROJECT_DIR / "kenlm" / "karakalpak_5gram.binary"

lm = kenlm.Model(str(LM_PATH))

test_words = [
    "tula",
    "bedenin",
    "tarap",
    "turıp",
    "basın",
    "mına",
    "jaqqa",
    "qızdırınıp",
    "atır",
    "tawıqlar"
]

print("KENLM VOCABULARY CHECK")
print("=" * 40)

for word in test_words:
    print(f"{word:20} -> {word in lm}")

KENLM VOCABULARY CHECK
tula                 -> False
bedenin              -> False
tarap                -> False
turıp                -> True
basın                -> False
mına                 -> True
jaqqa                -> False
qızdırınıp           -> False
atır                 -> True
tawıqlar             -> False


In [ ]:
import kenlm
from collections import Counter

LM_PATH = PROJECT_DIR / "kenlm" / "karakalpak_5gram.binary"

lm = kenlm.Model(str(LM_PATH))

all_words = []

for text in eval_df["text"]:
    words = str(text).lower().split()
    all_words.extend(words)

word_counts = Counter(all_words)

known_words = [
    word for word in all_words
    if word in lm
]

unknown_words = [
    word for word in all_words
    if word not in lm
]

oov_rate = len(unknown_words) / len(all_words) * 100

print("=" * 50)
print("MENDELEY KENLM VOCABULARY COVERAGE")
print("=" * 50)

print("Total words:", len(all_words))
print("Unique words:", len(word_counts))
print("Known word occurrences:", len(known_words))
print("OOV word occurrences:", len(unknown_words))

print(f"\nOOV Rate: {oov_rate:.2f}%")

print("\nTOP 20 OOV WORDS:")

for word, count in Counter(unknown_words).most_common(20):
    print(f"{word:25} {count}")

MENDELEY KENLM VOCABULARY COVERAGE
Total words: 26573
Unique words: 12152
Known word occurrences: 12348
OOV word occurrences: 14225

OOV Rate: 53.53%

TOP 20 OOV WORDS:
búl                       106
osı                       104
abai                      43
ǵoi                       42
qarai                     34
úsin                      34
bop                       28
boiınsa                   28
keiin                     25
siyaqtı                   24
uśın                      23
almai                     22
pen                       22
nemese                    21
kezde                     21
ús                        20
turalı                    20
júr                       19
au                        19
kep                       18


In [42]:
import torch
import soundfile as sf
import numpy as np

def transcribe_kenlm(audio_path):
    audio, sr = sf.read(
        str(audio_path),
        dtype="float32"
    )

    assert sr == 16000
    assert audio.ndim == 1

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.inference_mode():
        logits = model(
            input_values=inputs.input_values.to(device)
        ).logits

    logits_np = logits[0].float().cpu().numpy()

    prediction = decoder.decode(
        logits_np,
        beam_width=200
    )

    return prediction

In [43]:
from tqdm.auto import tqdm
from jiwer import wer, cer
import time
import pandas as pd

kenlm_references = []
kenlm_predictions = []
kenlm_results = []

start_time = time.time()

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Mendeley KenLM Evaluation"
):
    audio_path = EVAL_DIR / row["audio_path"]

    reference = str(row["text"]).lower().strip()

    prediction = transcribe_kenlm(
        audio_path
    ).lower().strip()

    kenlm_references.append(reference)
    kenlm_predictions.append(prediction)

    kenlm_results.append({
        "audio_path": row["audio_path"],
        "duration": row["duration"],
        "reference": reference,
        "prediction": prediction,
        "wer": wer(reference, prediction) * 100,
        "cer": cer(reference, prediction) * 100
    })

kenlm_seconds = time.time() - start_time

kenlm_wer = wer(
    kenlm_references,
    kenlm_predictions
) * 100

kenlm_cer = cer(
    kenlm_references,
    kenlm_predictions
) * 100

print("=" * 50)
print("MENDELEY KENLM EVALUATION")
print("=" * 50)

print(f"Samples: {len(kenlm_references)}")
print(f"KenLM WER: {kenlm_wer:.4f}%")
print(f"KenLM CER: {kenlm_cer:.4f}%")
print(f"Time: {kenlm_seconds:.2f}s")

Mendeley KenLM Evaluation:   0%|          | 0/1618 [00:00<?, ?it/s]

MENDELEY KENLM EVALUATION
Samples: 1618
KenLM WER: 73.8419%
KenLM CER: 17.3519%
Time: 370.51s


In [44]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# KenLM predictions
pd.DataFrame(kenlm_results).to_csv(
    RESULTS_DIR / "mendeley_step8500_kenlm_predictions.csv",
    index=False,
    encoding="utf-8"
)

# Greedy predictions
pd.DataFrame(results).to_csv(
    RESULTS_DIR / "mendeley_step8500_greedy_predictions.csv",
    index=False,
    encoding="utf-8"
)

# Comparison
comparison = pd.DataFrame([
    {
        "checkpoint": 8500,
        "decoder": "Greedy",
        "samples": 1618,
        "wer": total_wer,
        "cer": total_cer,
        "seconds": evaluation_seconds
    },
    {
        "checkpoint": 8500,
        "decoder": "KenLM",
        "samples": 1618,
        "wer": kenlm_wer,
        "cer": kenlm_cer,
        "seconds": kenlm_seconds
    }
])

comparison.to_csv(
    RESULTS_DIR / "mendeley_step8500_decoder_comparison.csv",
    index=False
)

display(comparison)

,checkpoint,decoder,samples,wer,cer,seconds
0,8500,Greedy,1618,59.831408,13.409962,157.397262
1,8500,KenLM,1618,73.841870,17.351922,370.505482


In [45]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

RESULTS_DIR = PROJECT_DIR / "results"

PREDICTIONS_PATH = (
    RESULTS_DIR / "mendeley_step8500_greedy_predictions.csv"
)

df = pd.read_csv(PREDICTIONS_PATH)

print("=" * 50)
print("MENDELEY ERROR ANALYSIS")
print("=" * 50)

print("Total samples:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

MENDELEY ERROR ANALYSIS
Total samples: 1618
Columns: ['audio_path', 'duration', 'reference', 'prediction', 'wer', 'cer']


,audio_path,duration,reference,prediction,wer,cer
0,Mendeley_train_audio/mendeley_audio_000002.wav,8.684,ústinde turıp basın ana jaqqa bir mına jaqqa b...,ústinde turıpbasın ana jaqqa br mınajaqqa bir ...,42.857143,8.130081
1,Mendeley_train_audio/mendeley_audio_000003.wav,8.164,dógereginde hámme jerde órip júr hám arasında ...,dógereginde hámmi jeti órip júr hám arasında s...,44.444444,12.096774
2,Mendeley_train_audio/mendeley_audio_000004.wav,9.724,oynaydı hámme qosıq aytadı quslar sayraydı jaw...,oynayda hámmin qosı aytadı quslar sayraydı jaw...,64.705882,9.917355
3,Mendeley_train_audio/mendeley_audio_000005.wav,8.788,titirkendirip jibergeni menen de báribir oǵan ...,tirkentiriw jibergeni menende bárli bir oǵan b...,64.705882,17.213115
4,Mendeley_train_audio/mendeley_audio_000006.wav,8.450,śaǵılısqan gúmis tamśılar marjan monśaqlar kib...,shaǵılısqan gúmiz tamshılar marjan monshaqlar ...,64.705882,14.516129


In [46]:
from jiwer import process_words
from collections import Counter

substitutions = Counter()
deletions = Counter()
insertions = Counter()

total_S = 0
total_D = 0
total_I = 0

for _, row in df.iterrows():

    reference = str(row["reference"]).lower().split()
    prediction = str(row["prediction"]).lower().split()

    result = process_words(
        " ".join(reference),
        " ".join(prediction)
    )

    total_S += result.substitutions
    total_D += result.deletions
    total_I += result.insertions

    for chunk in result.alignments[0]:

        ref_words = reference[
            chunk.ref_start_idx:chunk.ref_end_idx
        ]

        pred_words = prediction[
            chunk.hyp_start_idx:chunk.hyp_end_idx
        ]

        if chunk.type == "substitute":
            for ref_word, pred_word in zip(ref_words, pred_words):
                substitutions[(ref_word, pred_word)] += 1

        elif chunk.type == "delete":
            for word in ref_words:
                deletions[word] += 1

        elif chunk.type == "insert":
            for word in pred_words:
                insertions[word] += 1


print("=" * 45)
print("WORD ERROR ANALYSIS")
print("=" * 45)

print("Substitutions:", total_S)
print("Deletions:", total_D)
print("Insertions:", total_I)

print("\nTOP 20 SUBSTITUTIONS:")
for (ref, pred), count in substitutions.most_common(20):
    print(f"{ref:20} -> {pred:20} : {count}")

print("\nTOP 20 DELETIONS:")
for word, count in deletions.most_common(20):
    print(f"{word:25} : {count}")

print("\nTOP 20 INSERTIONS:")
for word, count in insertions.most_common(20):
    print(f"{word:25} : {count}")

WORD ERROR ANALYSIS
Substitutions: 13785
Deletions: 1521
Insertions: 593

TOP 20 SUBSTITUTIONS:
búl                  -> bul                  : 71
qarai                -> qaray                : 28
abai                 -> abay                 : 25
boiınsa              -> boyınsha             : 24
uśın                 -> ushın                : 16
túrǵan               -> turǵan               : 15
qaita                -> qayta                : 14
júmıs                -> jumıs                : 14
almai                -> almay                : 13
boiı                 -> boyı                 : 13
sıqqan               -> shıqqan              : 13
ús                   -> úsh                  : 12
qandai               -> qanday               : 12
birnese              -> birneshe             : 12
aitqan               -> aytqan               : 11
abaidıń              -> abaydıń              : 11
aitıp                -> aytıp                : 11
auıl                 -> awıl                 : 11
su  

In [47]:
import re
import unicodedata

from jiwer import wer, cer

def diagnostic_normalize(text):
    text = unicodedata.normalize("NFC", str(text).lower())

    # Faqat tajriba uchun tanlangan so'z variantlari
    word_mapping = {
        "búl": "bul",
        "qarai": "qaray",
        "abai": "abay",
        "boiınsa": "boyınsha",
        "uśın": "ushın",
        "qaita": "qayta",
        "almai": "almay",
        "qandai": "qanday",
        "birnese": "birneshe",
        "aitqan": "aytqan",
        "aitıp": "aytıp",
    }

    words = text.split()

    words = [
        word_mapping.get(word, word)
        for word in words
    ]

    return " ".join(words)

In [48]:
original_refs = df["reference"].astype(str).tolist()
original_preds = df["prediction"].astype(str).tolist()

normalized_refs = [
    diagnostic_normalize(text)
    for text in original_refs
]

normalized_preds = [
    diagnostic_normalize(text)
    for text in original_preds
]

original_wer = wer(original_refs, original_preds) * 100
original_cer = cer(original_refs, original_preds) * 100

normalized_wer = wer(normalized_refs, normalized_preds) * 100
normalized_cer = cer(normalized_refs, normalized_preds) * 100

print("=" * 50)
print("NORMALIZATION IMPACT ANALYSIS")
print("=" * 50)

print(f"Original WER:   {original_wer:.4f}%")
print(f"Normalized WER: {normalized_wer:.4f}%")

print()

print(f"Original CER:   {original_cer:.4f}%")
print(f"Normalized CER: {normalized_cer:.4f}%")

print()

print(f"WER difference: {original_wer - normalized_wer:.4f} pp")
print(f"CER difference: {original_cer - normalized_cer:.4f} pp")

NORMALIZATION IMPACT ANALYSIS
Original WER:   59.8314%
Normalized WER: 58.7852%

Original CER:   13.4100%
Normalized CER: 13.1934%

WER difference: 1.0462 pp
CER difference: 0.2166 pp


In [49]:
from collections import Counter

vocab = processor.tokenizer.get_vocab()

allowed_chars = {
    token
    for token in vocab
    if len(token) == 1 and token != "|"
}

all_text = " ".join(
    eval_df["text"].astype(str).str.lower()
)

unknown_chars = Counter(
    char for char in all_text
    if not char.isspace() and char not in allowed_chars
)

print("=" * 50)
print("CHARACTER VOCABULARY ANALYSIS")
print("=" * 50)

print("Model vocabulary size:", len(vocab))

print("\nUNKNOWN CHARACTERS:")

for char, count in unknown_chars.most_common():
    print(f"{repr(char):10} -> {count}")

print("\nTotal unknown character occurrences:",
      sum(unknown_chars.values()))

CHARACTER VOCABULARY ANALYSIS
Model vocabulary size: 36

UNKNOWN CHARACTERS:
'ś'        -> 626
'ć'        -> 18
'һ'        -> 15

Total unknown character occurrences: 659


In [50]:
from collections import Counter

unknown_chars = ["ś", "ć", "һ"]

for char in unknown_chars:

    words = Counter(
        word
        for text in eval_df["text"].astype(str).str.lower()
        for word in text.split()
        if char in word
    )

    print(f"\n{'=' * 45}")
    print(f"CHARACTER: {char}")
    print(f"TOTAL OCCURRENCES: {sum(words.values())}")
    print(f"UNIQUE WORDS: {len(words)}")
    print("=" * 45)

    for word, count in words.most_common(20):
        print(f"{word:25} {count}")


CHARACTER: ś
TOTAL OCCURRENCES: 594
UNIQUE WORDS: 411
uśın                      23
iśinde                    8
śıqtı                     8
úś                        8
birneśe                   8
boyınśa                   7
heś                       6
quyaś                     5
baslawıś                  5
birinśi                   5
oqıwśılar                 5
śekem                     5
bayanlawıś                4
baǵındırıwśı              4
keśke                     4
kiśkene                   4
aśıp                      4
śıǵıp                     4
śıqqan                    4
neśe                      4

CHARACTER: ć
TOTAL OCCURRENCES: 18
UNIQUE WORDS: 16
vladimirović              3
ćemodanımdı               1
avtorućka                 1
poćta                     1
manćjur                   1
rısqúlović                1
ćili                      1
ućaskelerge               1
ćernov                    1
tućei                     1
otćayannaya               1
felićedegi             

In [51]:
import unicodedata
from jiwer import wer, cer

def normalize_karakalpak(text):
    text = unicodedata.normalize(
        "NFC",
        str(text).lower()
    )

    text = text.replace("ś", "sh")
    text = text.replace("ć", "ch")

    return " ".join(text.split())

In [52]:
references_original = df["reference"].astype(str).tolist()
predictions_original = df["prediction"].astype(str).tolist()

references_normalized = [
    normalize_karakalpak(text)
    for text in references_original
]

predictions_normalized = [
    normalize_karakalpak(text)
    for text in predictions_original
]

print("=" * 50)
print("CHARACTER NORMALIZATION IMPACT")
print("=" * 50)

print(
    "Original WER:",
    round(wer(references_original, predictions_original) * 100, 4)
)

print(
    "Normalized WER:",
    round(wer(references_normalized, predictions_normalized) * 100, 4)
)

print(
    "Original CER:",
    round(cer(references_original, predictions_original) * 100, 4)
)

print(
    "Normalized CER:",
    round(cer(references_normalized, predictions_normalized) * 100, 4)
)

CHARACTER NORMALIZATION IMPACT
Original WER: 59.8314
Normalized WER: 58.7965
Original CER: 13.41
Normalized CER: 12.7472


In [53]:
from collections import Counter
from jiwer import process_characters

char_substitutions = Counter()
char_deletions = Counter()
char_insertions = Counter()

for ref, pred in zip(
    df["reference"].astype(str),
    df["prediction"].astype(str)
):

    result = process_characters(ref, pred)

    ref_chars = list(ref)
    pred_chars = list(pred)

    for chunk in result.alignments[0]:

        if chunk.type == "substitute":
            r = ref_chars[
                chunk.ref_start_idx:chunk.ref_end_idx
            ]

            p = pred_chars[
                chunk.hyp_start_idx:chunk.hyp_end_idx
            ]

            for a, b in zip(r, p):
                char_substitutions[(a, b)] += 1

        elif chunk.type == "delete":
            for char in ref_chars[
                chunk.ref_start_idx:chunk.ref_end_idx
            ]:
                char_deletions[char] += 1

        elif chunk.type == "insert":
            for char in pred_chars[
                chunk.hyp_start_idx:chunk.hyp_end_idx
            ]:
                char_insertions[char] += 1

print("=" * 50)
print("TOP 25 CHARACTER SUBSTITUTIONS")
print("=" * 50)

for (ref, pred), count in char_substitutions.most_common(25):
    print(f"{repr(ref):8} -> {repr(pred):8} : {count}")

print("\nTOP 15 DELETED CHARACTERS")

for char, count in char_deletions.most_common(15):
    print(f"{repr(char):8} : {count}")

print("\nTOP 15 INSERTED CHARACTERS")

for char, count in char_insertions.most_common(15):
    print(f"{repr(char):8} : {count}")

TOP 25 CHARACTER SUBSTITUTIONS
'i'      -> 'y'      : 2220
'u'      -> 'w'      : 1044
'ú'      -> 'u'      : 963
'ś'      -> 'h'      : 537
'ń'      -> 'n'      : 452
'i'      -> 'e'      : 412
'e'      -> 'i'      : 366
'ı'      -> 'a'      : 355
'i'      -> 'ı'      : 267
'a'      -> 'ı'      : 258
'á'      -> 'a'      : 203
'd'      -> 't'      : 170
'o'      -> 'a'      : 149
'ú'      -> 'i'      : 146
'h'      -> 'x'      : 133
'a'      -> 'á'      : 129
'p'      -> 'b'      : 120
'ú'      -> 'ı'      : 116
'ı'      -> 'i'      : 111
'u'      -> 'ı'      : 109
'd'      -> 'l'      : 107
's'      -> 'h'      : 100
'ó'      -> 'ú'      : 93
'o'      -> 'u'      : 92
'ı'      -> 'u'      : 87

TOP 15 DELETED CHARACTERS
' '      : 1522
'i'      : 888
'ı'      : 628
'n'      : 274
'e'      : 219
't'      : 186
'd'      : 182
'a'      : 166
'q'      : 161
'l'      : 149
's'      : 140
'ń'      : 135
'y'      : 127
'ú'      : 124
'u'      : 121

TOP 15 INSERTED CHARACTERS
'h'      : 146

In [54]:
from collections import Counter
from jiwer import process_words

iy_pairs = Counter()

for _, row in df.iterrows():

    ref = str(row["reference"])
    pred = str(row["prediction"])

    alignment = process_words(ref, pred)

    ref_words = ref.split()
    pred_words = pred.split()

    for chunk in alignment.alignments[0]:

        if chunk.type != "substitute":
            continue

        r = ref_words[
            chunk.ref_start_idx:chunk.ref_end_idx
        ]

        p = pred_words[
            chunk.hyp_start_idx:chunk.hyp_end_idx
        ]

        for rw, pw in zip(r, p):

            # Faqat i -> y farqi mavjud bo'lgan so'zlar
            if rw != pw and rw.replace("i", "y") == pw:
                iy_pairs[(rw, pw)] += 1

print("TOP i -> y WORD VARIANTS")
print("=" * 50)

for (ref, pred), count in iy_pairs.most_common(30):
    print(f"{ref:20} -> {pred:20} : {count}")

TOP i -> y WORD VARIANTS
qarai                -> qaray                : 28
abai                 -> abay                 : 25
qaita                -> qayta                : 14
almai                -> almay                : 13
boiı                 -> boyı                 : 13
qandai               -> qanday               : 12
aitqan               -> aytqan               : 11
abaidıń              -> abaydıń              : 11
aitıp                -> aytıp                : 11
abaiǵa               -> abayǵa               : 10
qaida                -> qayda                : 9
qoiıp                -> qoyıp                : 9
ǵoi                  -> ǵoy                  : 9
qaitıp               -> qaytıp               : 9
qalai                -> qalay                : 8
osındai              -> osınday              : 8
jaiın                -> jayın                : 8
arnaiı               -> arnayı               : 7
oiın                 -> oyın                 : 6
aita                 -> ayta      